# 6.4 — Backpropagation

Backpropagation is the bookkeeping method that lets a neural network learn: run a forward pass, cache the local values each operation needs, then send the loss gradient backward by multiplying local derivatives along the computational path. In this lesson, every network is built from scratch in NumPy so the chain rule, shapes, updates, softmax comparisons, scale, and memory costs stay visible instead of becoming framework magic.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build backpropagation one idea at a time. Run each cell in order and read the printed intermediate values — every local derivative, shape, and update is shown so nothing is a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, matrix products, and explicit derivative bookkeeping.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for small synthetic examples.

### 1. Forward pass caches local values

A neural network is a composed function. Even one hidden unit multiplies inputs by weights, adds a bias, gates the result with ReLU, then passes the activation onward. Backprop works because the forward pass remembers `x`, `z`, and `h`; those cached values determine the local derivatives used later.

In [ ]:
x_w = np.array([1.5, -0.5])
w_w = np.array([1.8, -0.7])
b_w = 0.7
z_w = float(w_w @ x_w + b_w)
h_w = max(0.0, z_w)
print("z:", round(z_w, 3), "h:", round(h_w, 3))
assert round(z_w, 3) == 3.750 and round(h_w, 3) == 3.750

▶ What you'll see: the affine signal is 3.750 and ReLU passes it through because it is positive.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["w0*x0", "w1*x1", "bias", "z", "h"], [w_w[0]*x_w[0], w_w[1]*x_w[1], b_w, z_w, h_w], color=["teal", "teal", "orange", "purple", "green"])
plt.title("1: forward pass pieces"); plt.ylabel("value"); plt.xticks(rotation=20); plt.show()

▶ What you'll see: the pre-activation is the sum of two weighted input contributions plus a bias.

*Why it's done this way:* the final loss alone does not tell us whether a ReLU was open or which input created a weight's contribution. Caching the local forward values makes each derivative cheap and exact during the backward pass.

### 2. Chain rule sends credit backward

Suppose the hidden activation feeds a scalar output `y_hat = v h`, and the loss is half-squared error. Backprop starts with $\partial L/\partial \hat y$ and repeatedly multiplies by local derivatives until each earlier parameter receives credit.

In [ ]:
v_w = 0.8
y_w = 2.0
yhat_w = v_w * h_w
loss_w = 0.5 * (yhat_w - y_w) ** 2
print("y_hat:", round(yhat_w, 3), "loss:", round(loss_w, 3))
assert round(yhat_w, 3) == 3.000 and round(loss_w, 3) == 0.500

▶ What you'll see: the model over-predicts 2.0 with 3.0, creating loss 0.5.

In [ ]:
dL_dyhat_w = yhat_w - y_w
dL_dh_w = dL_dyhat_w * v_w
dL_dz_w = dL_dh_w * (1.0 if z_w > 0 else 0.0)
dL_dw_w = dL_dz_w * x_w
dL_db_w = dL_dz_w
print("dL/dh:", round(dL_dh_w, 3), "dL/dw:", np.round(dL_dw_w, 3), "dL/db:", round(dL_db_w, 3))
assert np.allclose(np.round(dL_dw_w, 3), [1.2, -0.4])

▶ What you'll see: the same upstream error becomes different weight gradients because the input coordinates differ.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["dL/dyhat", "dL/dh", "dL/dz", "dL/dw0", "dL/dw1"], [dL_dyhat_w, dL_dh_w, dL_dz_w, dL_dw_w[0], dL_dw_w[1]], color="crimson")
plt.axhline(0, color="black", linewidth=0.8); plt.title("2: gradient signal moving backward"); plt.xticks(rotation=25); plt.show()

▶ What you'll see: one upstream error is redistributed into signed parameter gradients.

*Why it's done this way:* the chain rule says a parameter only affects the loss through paths downstream of it. Multiplying local derivatives decomposes a large network derivative into small reusable pieces.

### 3. Shapes make layer backprop vectorized

For a batch, `X` is examples×features and `W1` is features×hidden, so `Z1 = X @ W1 + b1` is examples×hidden. The backward formulas mirror those shapes: `dW1 = X.T @ dZ1`, `db1 = sum(dZ1)`, and `dX = dZ1 @ W1.T`.

In [ ]:
X_w = np.array([[1.5, -0.5], [0.2, 1.0], [-1.0, 0.5]])
W1_w = np.array([[1.8, -0.3], [-0.7, 0.9]])
b1_w = np.array([0.7, -0.2])
Z1_w = X_w @ W1_w + b1_w
H1_w = np.maximum(0, Z1_w)
print("Z1 shape:", Z1_w.shape, "H1 shape:", H1_w.shape)
assert Z1_w.shape == (3, 2)

▶ What you'll see: one hidden vector is produced for each of the three examples.

In [ ]:
dH1_w = np.array([[0.8, -0.1], [0.2, 0.4], [-0.3, 0.6]])
dZ1_w = dH1_w * (Z1_w > 0)
dW1_w = X_w.T @ dZ1_w
db1_w = np.sum(dZ1_w, axis=0)
dX_w = dZ1_w @ W1_w.T
print("dW1 shape:", dW1_w.shape, "db1:", np.round(db1_w, 3), "dX shape:", dX_w.shape)
assert dW1_w.shape == W1_w.shape and dX_w.shape == X_w.shape

▶ What you'll see: every gradient has the same shape as the quantity it updates or passes backward to.

In [ ]:
plt.figure(figsize=(4.2, 3.2)); plt.imshow(dZ1_w, cmap="coolwarm", aspect="auto"); plt.colorbar(label="dL/dZ1")
plt.title("3: ReLU gates in the batch gradient"); plt.xlabel("hidden unit"); plt.ylabel("example"); plt.show()

▶ What you'll see: entries where the ReLU was closed are exactly zero in the pre-activation gradient.

*Why it's done this way:* vectorized backprop is scalar chain rule stacked into matrices. The transpose in `X.T @ dZ1` appears because each weight gradient sums input-feature times upstream-gradient products over examples.

### 4. Updates turn gradients into learning

A gradient points in the direction that increases loss. Gradient descent subtracts it. With parameter 2.000, learning rate 0.090, and gradient 1.800, the new value is 1.838.

In [ ]:
theta_w = 2.0
g_w = 1.8
eta_w = 0.09
theta_new_w = theta_w - eta_w * g_w
print("theta before:", theta_w, "theta after:", round(theta_new_w, 3))
assert round(theta_new_w, 3) == 1.838

▶ What you'll see: the parameter moves by 0.162, not by a huge jump.

In [ ]:
theta_grid_w = np.linspace(0.5, 3.0, 120)
loss_grid_w = (theta_grid_w - 1.0) ** 2
plt.figure(figsize=(4.4, 3)); plt.plot(theta_grid_w, loss_grid_w, color="navy")
plt.scatter([theta_w, theta_new_w], [(theta_w-1)**2, (theta_new_w-1)**2], color=["red", "green"])
plt.title("4: one descent step on a loss bowl"); plt.xlabel("parameter"); plt.ylabel("loss"); plt.show()

▶ What you'll see: the green point lands lower on the bowl than the red point.

*Why it's done this way:* each gradient is local to the current parameters and often estimated from a minibatch. Small repeated nudges let the model remeasure the slope after every change.

### 5. Softmax turns scores into comparable gradients

Classification networks usually output logits, not probabilities. Softmax exponentiates and normalizes logits, while cross-entropy creates the particularly simple gradient `probabilities - target`.

In [ ]:
logits_w = np.array([3.75, 0.40])
exp_w = np.exp(logits_w - np.max(logits_w))
probs_w = exp_w / np.sum(exp_w)
print("softmax probabilities:", np.round(probs_w, 3))
assert round(float(probs_w[0]), 3) == 0.966

▶ What you'll see: the 3.750 logit receives about 96.6% probability against the 0.400 baseline.

In [ ]:
target_w = np.array([1.0, 0.0])
ce_w = -np.sum(target_w * np.log(probs_w))
dlogits_w = probs_w - target_w
print("cross entropy:", round(float(ce_w), 3), "dL/dlogits:", np.round(dlogits_w, 3))
assert round(float(ce_w), 3) == 0.034

▶ What you'll see: the loss is small but still produces a corrective gradient.

In [ ]:
plt.figure(figsize=(4.4, 3)); plt.bar(["class 0", "class 1"], probs_w, color=["green", "gray"])
plt.ylim(0, 1); plt.title("5: softmax probabilities"); plt.ylabel("probability"); plt.show()

▶ What you'll see: probabilities sum to 1, making raw scores into a calibrated comparison.

*Why it's done this way:* absolute logits do not define a class decision by themselves. Softmax creates a differentiable competition, and cross-entropy turns the competition into a gradient that moves probability mass toward the target.

### 6. Scale and memory shape practical backprop

Backprop is exact chain-rule arithmetic, but training quality depends on scale and hardware. Normalization changes effective gradient scale, and storing forward activations costs memory because the backward pass needs them.

In [ ]:
signal_w = 3.75
mean_w = 1.0
var_w = 0.25
eps_w = 1e-5
normed_w = (signal_w - mean_w) / np.sqrt(var_w + eps_w)
print("normalized value:", round(float(normed_w), 3))
assert round(float(normed_w), 3) == 5.500

▶ What you'll see: the activation is 5.5 standard deviations above the mean.

In [ ]:
vectors_w = 2
width_w = 128
memory_kb_w = vectors_w * width_w * 4 / 1024
print("activation memory KB:", round(memory_kb_w, 3))
assert round(memory_kb_w, 3) == 1.000

▶ What you'll see: two length-128 float32 activation vectors use exactly 1 KB.

In [ ]:
layers_w = np.arange(1, 9)
mem_curve_w = layers_w * 32 * 128 * 4 / 1024
plt.figure(figsize=(4.4, 3)); plt.plot(layers_w, mem_curve_w, marker="o", color="purple")
plt.title("6: cached activations grow with depth"); plt.xlabel("layers"); plt.ylabel("activation memory (KB)"); plt.show()

▶ What you'll see: cached activation memory grows linearly in this simplified depth sweep.

*Why it's done this way:* the chain rule multiplies local derivatives, so scale controls whether gradients stay useful. Caching makes exact gradients efficient but turns depth, width, and batch size into memory pressure.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, matrix products, activations, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for the heatmaps, bars, and learning curves used to inspect backprop.
np.random.seed(0) # make random initialization and toy data reproducible.

def relu(z): # define the ReLU gate used by scratch neural networks.
    return np.maximum(0, z) # keep positive pre-activations and zero out negative ones.

def relu_grad(z): # define the local derivative of ReLU with respect to its input.
    return (z > 0).astype(float) # return 1 on open gates and 0 on closed gates.

def softmax(logits): # define a numerically stable softmax for class probabilities.
    shifted = logits - np.max(logits, axis=-1, keepdims=True) # subtract max without changing probabilities.
    exp_vals = np.exp(shifted) # exponentiate shifted logits.
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True) # normalize each row to sum to 1.

def mse_loss(yhat, y): # define half mean-squared error so the derivative is simple.
    return 0.5 * np.mean((yhat - y) ** 2) # average over examples and keep the 1/2 convention.

## 🟢 Basics (warm-up)

### Basic 1 — Compute one affine pre-activation

**Goal.** Build the first number in a forward pass, because every backprop derivative later depends on cached affine pieces. We build it in one small inspectable cell.

In [ ]:
x_b1 = np.array([1.5, -0.5]) # store two input features for one example.
w_b1 = np.array([1.8, -0.7]) # store the two weights feeding one neuron.
b_b1 = 0.7 # store the neuron bias.
z_b1 = float(w_b1 @ x_b1 + b_b1) # compute w dot x plus bias.
print("z_b1:", round(z_b1, 3)) # inspect the pre-activation.
assert round(z_b1, 3) == 3.750 # verify the lesson arithmetic.
plt.figure(figsize=(4, 3)); plt.bar(["w0*x0", "w1*x1", "bias"], [w_b1[0]*x_b1[0], w_b1[1]*x_b1[1], b_b1], color="teal")
plt.title("Basic 1: affine pieces"); plt.ylabel("contribution"); plt.show()

▶ What you'll see: 2.700 + 0.350 + 0.700 sums to 3.750.

👀 Takeaway: an affine neuron is a weighted sum plus a bias, and those pieces must be cached for gradients.

### Basic 2 — Apply a ReLU gate

**Goal.** Gate a pre-activation with ReLU, because the backward pass will either pass gradient through or stop it. We build it in one small inspectable cell.

In [ ]:
z_b2 = np.array([3.75, -1.2, 0.0, 2.1]) # create pre-activations with positive, negative, and zero cases.
h_b2 = relu(z_b2) # apply the ReLU activation elementwise.
gate_b2 = relu_grad(z_b2) # compute local derivatives of the ReLU gate.
print("h_b2:", h_b2, "gate_b2:", gate_b2) # inspect outputs and gradients.
assert np.array_equal(gate_b2, np.array([1., 0., 0., 1.])) # verify the gates.
plt.figure(figsize=(4, 3)); plt.bar(range(len(z_b2)), gate_b2, color="orange")
plt.title("Basic 2: ReLU derivative gate"); plt.ylabel("local derivative"); plt.show()

▶ What you'll see: only units with positive pre-activation pass gradient backward.

👀 Takeaway: ReLU stores a simple local derivative: open gate = 1, closed gate = 0.

### Basic 3 — Compute a scalar loss

**Goal.** Turn a prediction error into a loss, because backprop starts from the derivative of that loss. We build it in one small inspectable cell.

In [ ]:
yhat_b3 = 3.0 # define a scalar prediction from a tiny network.
y_b3 = 2.0 # define the target value.
err_b3 = yhat_b3 - y_b3 # compute prediction minus target.
loss_b3 = 0.5 * err_b3 ** 2 # compute half-squared error.
dL_dyhat_b3 = err_b3 # derivative of half-squared error.
print("loss_b3:", round(loss_b3, 3), "dL/dyhat:", round(dL_dyhat_b3, 3))
assert round(loss_b3, 3) == 0.500 and round(dL_dyhat_b3, 3) == 1.000
plt.figure(figsize=(4, 3)); plt.bar(["error", "loss", "gradient"], [err_b3, loss_b3, dL_dyhat_b3], color="purple")
plt.title("Basic 3: loss starts backprop"); plt.show()

▶ What you'll see: half-squared loss is 0.5 and its derivative is the signed error.

👀 Takeaway: backprop begins at the loss, where the first gradient is usually easy to compute.

### Basic 4 — Chain through one output weight

**Goal.** Backpropagate from a scalar prediction to the hidden activation and output weight. We build it in one small inspectable cell.

In [ ]:
h_b4 = 3.75 # cached hidden activation.
v_b4 = 0.8 # output layer weight.
y_b4 = 2.0 # target value.
yhat_b4 = v_b4 * h_b4 # compute scalar prediction.
dL_dyhat_b4 = yhat_b4 - y_b4 # derivative of half-squared error.
dL_dv_b4 = dL_dyhat_b4 * h_b4 # local derivative with respect to v is h.
dL_dh_b4 = dL_dyhat_b4 * v_b4 # local derivative with respect to h is v.
print("dL/dv:", round(dL_dv_b4, 3), "dL/dh:", round(dL_dh_b4, 3))
assert round(dL_dv_b4, 3) == 3.750 and round(dL_dh_b4, 3) == 0.800
plt.figure(figsize=(4, 3)); plt.bar(["dL/dv", "dL/dh"], [dL_dv_b4, dL_dh_b4], color="crimson")
plt.title("Basic 4: output-layer gradients"); plt.ylabel("gradient"); plt.show()

▶ What you'll see: the output weight gets the upstream error multiplied by the hidden activation.

👀 Takeaway: each parameter receives the upstream gradient times its own local derivative.

### Basic 5 — Backpropagate into input weights

**Goal.** Continue the same gradient through ReLU and the affine map. We build it in one small inspectable cell.

In [ ]:
x_b5 = np.array([1.5, -0.5]) # cached input vector.
z_b5 = 3.75 # cached pre-activation.
dL_dh_b5 = 0.8 # upstream gradient from the output layer.
dL_dz_b5 = dL_dh_b5 * (1.0 if z_b5 > 0 else 0.0) # pass through ReLU gate.
dL_dw_b5 = dL_dz_b5 * x_b5 # affine weight derivative equals input times upstream gradient.
dL_db_b5 = dL_dz_b5 # bias derivative equals upstream pre-activation gradient.
print("dL/dw:", np.round(dL_dw_b5, 3), "dL/db:", round(dL_db_b5, 3))
assert np.allclose(np.round(dL_dw_b5, 3), [1.2, -0.4])
plt.figure(figsize=(4, 3)); plt.bar(["w0", "w1", "b"], [dL_dw_b5[0], dL_dw_b5[1], dL_db_b5], color=["teal", "teal", "orange"])
plt.axhline(0, color="black", linewidth=0.8); plt.title("Basic 5: affine gradients"); plt.show()

▶ What you'll see: the second weight gradient is negative because the second input feature is negative.

👀 Takeaway: the same upstream error can push different weights in different directions because inputs differ.

### Basic 6 — Take one gradient descent step

**Goal.** Update a parameter from its gradient, because backprop only becomes learning when gradients move weights. We build it in one small inspectable cell.

In [ ]:
theta_b6 = 2.0 # define one scalar parameter.
grad_b6 = 1.8 # define its gradient.
eta_b6 = 0.09 # define the learning rate.
theta_new_b6 = theta_b6 - eta_b6 * grad_b6 # subtract the scaled gradient.
print("after:", round(theta_new_b6, 3)) # inspect the updated parameter.
assert round(theta_new_b6, 3) == 1.838 # verify 2 - .09*1.8.
plt.figure(figsize=(4, 3)); plt.bar(["before", "after"], [theta_b6, theta_new_b6], color=["gray", "green"])
plt.title("Basic 6: one parameter update"); plt.ylabel("parameter value"); plt.show()

▶ What you'll see: the parameter decreases by 0.162 because the gradient is positive.

👀 Takeaway: gradient descent moves opposite the gradient by a learning-rate-controlled amount.

### Basic 7 — Softmax two logits

**Goal.** Convert two raw scores into probabilities, because classification losses compare scores through normalized probabilities. We build it in one small inspectable cell.

In [ ]:
logits_b7 = np.array([3.75, 0.40]) # define the lesson score and baseline score.
probs_b7 = softmax(logits_b7) # compute stable softmax probabilities.
print("probabilities:", np.round(probs_b7, 3)) # inspect normalized class probabilities.
assert round(float(probs_b7[0]), 3) == 0.966 # verify the lesson probability.
plt.figure(figsize=(4, 3)); plt.bar(["score 3.75", "baseline 0.40"], probs_b7, color=["green", "gray"])
plt.ylim(0, 1); plt.title("Basic 7: softmax comparison"); plt.ylabel("probability"); plt.show()

▶ What you'll see: the larger score receives about 96.6% probability.

👀 Takeaway: softmax turns arbitrary logits into a differentiable comparison that sums to 1.

### Basic 8 — Cross-entropy gradient

**Goal.** Compute the softmax-cross-entropy gradient, because it is the clean signal sent backward from a classifier. We build it in one small inspectable cell.

In [ ]:
probs_b8 = np.array([0.966, 0.034]) # use rounded probabilities from the two-logit comparison.
target_b8 = np.array([1.0, 0.0]) # encode class 0 as the correct class.
loss_b8 = -np.sum(target_b8 * np.log(probs_b8)) # compute one-example cross-entropy.
dlogits_b8 = probs_b8 - target_b8 # compute the gradient with respect to logits.
print("loss:", round(float(loss_b8), 3), "dlogits:", np.round(dlogits_b8, 3))
assert np.allclose(np.round(dlogits_b8, 3), [-0.034, 0.034])
plt.figure(figsize=(4, 3)); plt.bar(["class 0", "class 1"], dlogits_b8, color=["green", "red"])
plt.axhline(0, color="black", linewidth=0.8); plt.title("Basic 8: logits gradient"); plt.show()

▶ What you'll see: gradient descent will raise the correct logit and lower the incorrect one.

👀 Takeaway: for softmax plus cross-entropy, the backward signal is probability minus target.

### Basic 9 — Normalize one activation

**Goal.** Standardize a signal with mean and variance, because scale affects how gradients behave through deep compositions. We build it in one small inspectable cell.

In [ ]:
signal_b9 = 3.75 # define the activation value being normalized.
mean_b9 = 1.0 # define the reference mean.
var_b9 = 0.25 # define the reference variance.
eps_b9 = 1e-5 # define a small stabilizer.
norm_b9 = (signal_b9 - mean_b9) / np.sqrt(var_b9 + eps_b9) # compute normalized value.
print("normalized:", round(float(norm_b9), 3)) # inspect the standardized activation.
assert round(float(norm_b9), 3) == 5.500 # verify the lesson normalization number.
plt.figure(figsize=(4, 3)); plt.bar(["raw", "normalized"], [signal_b9, norm_b9], color=["gray", "purple"])
plt.title("Basic 9: normalization changes scale"); plt.ylabel("value"); plt.show()

▶ What you'll see: after dividing by a small standard deviation, the normalized value is 5.5.

👀 Takeaway: scale choices change the magnitude of signals that the chain rule will multiply.

### Basic 10 — Count activation memory

**Goal.** Compute the memory needed to cache activations, because backprop must remember forward values. We build it in one small inspectable cell.

In [ ]:
vectors_b10 = 2 # count cached activation vectors.
width_b10 = 128 # define each vector length.
bytes_b10 = 4 # use float32 storage.
memory_kb_b10 = vectors_b10 * width_b10 * bytes_b10 / 1024 # convert bytes to KB.
print("memory KB:", round(memory_kb_b10, 3)) # inspect activation storage.
assert round(memory_kb_b10, 3) == 1.000 # verify the 1 KB calculation.
plt.figure(figsize=(4, 3)); plt.bar(["activation cache"], [memory_kb_b10], color="slateblue")
plt.title("Basic 10: cached activation memory"); plt.ylabel("KB"); plt.show()

▶ What you'll see: even a tiny activation cache has a concrete byte cost.

👀 Takeaway: backprop trades memory for efficient exact gradients.

## 🟡 Easy

### Easy 1 — Train one linear neuron with backprop

**Goal.** train one linear neuron with backprop, because this is the next step after the scalar chain-rule warm-up. We build it in one compact runnable cell.

In [ ]:
X_e1 = np.array([[0.0], [1.0], [2.0], [3.0]]) # create one-feature training inputs.
y_e1 = np.array([[1.0], [3.0], [5.0], [7.0]]) # create targets from y = 2x + 1.
w_e1 = np.array([[0.0]]) # initialize the weight.
b_e1 = np.array([0.0]) # initialize the bias.
losses_e1 = [] # store training loss values.
for step_e1 in range(120): # run repeated full-batch gradient descent.
    pred_e1 = X_e1 @ w_e1 + b_e1 # forward pass.
    d_pred_e1 = (pred_e1 - y_e1) / len(X_e1) # averaged loss gradient.
    w_e1 -= 0.12 * (X_e1.T @ d_pred_e1) # update weight.
    b_e1 -= 0.12 * np.sum(d_pred_e1, axis=0) # update bias.
    losses_e1.append(float(mse_loss(X_e1 @ w_e1 + b_e1, y_e1))) # record loss.
print("trained w,b:", round(float(w_e1[0, 0]), 3), round(float(b_e1[0]), 3))
assert losses_e1[-1] < losses_e1[0]
plt.figure(figsize=(4.5, 3)); plt.plot(losses_e1, color="teal")
plt.title("Easy 1: linear neuron loss"); plt.xlabel("step"); plt.ylabel("half-MSE"); plt.show()

▶ What you'll see: the loss falls as the weight approaches 2 and the bias approaches 1.

👀 Takeaway: backprop for a linear neuron is matrix calculus plus gradient descent.

### Easy 2 — Train a tiny ReLU network

**Goal.** train a tiny relu network, because this is the next step after the scalar chain-rule warm-up. We build it in one compact runnable cell.

In [ ]:
X_e2 = np.linspace(-1.5, 1.5, 9).reshape(-1, 1) # create one-dimensional inputs.
y_e2 = (X_e2 ** 2 + 0.2).reshape(-1, 1) # create a curved regression target.
rng_e2 = np.random.default_rng(2) # create reproducible initialization.
W1_e2 = rng_e2.normal(scale=0.4, size=(1, 4)); b1_e2 = np.zeros(4) # initialize hidden layer.
W2_e2 = rng_e2.normal(scale=0.4, size=(4, 1)); b2_e2 = np.zeros(1) # initialize output layer.
losses_e2 = [] # store loss values.
for step_e2 in range(800): # run full-batch gradient descent.
    Z1_e2 = X_e2 @ W1_e2 + b1_e2; H1_e2 = relu(Z1_e2); yhat_e2 = H1_e2 @ W2_e2 + b2_e2 # forward pass.
    d_yhat_e2 = (yhat_e2 - y_e2) / len(X_e2) # loss derivative.
    dW2_e2 = H1_e2.T @ d_yhat_e2; db2_e2 = np.sum(d_yhat_e2, axis=0) # output gradients.
    dZ1_e2 = (d_yhat_e2 @ W2_e2.T) * relu_grad(Z1_e2) # hidden pre-activation gradient.
    dW1_e2 = X_e2.T @ dZ1_e2; db1_e2 = np.sum(dZ1_e2, axis=0) # hidden gradients.
    W2_e2 -= 0.08 * dW2_e2; b2_e2 -= 0.08 * db2_e2; W1_e2 -= 0.08 * dW1_e2; b1_e2 -= 0.08 * db1_e2 # update all parameters.
    if step_e2 % 10 == 0: losses_e2.append(float(mse_loss(yhat_e2, y_e2))) # record loss.
yhat_e2 = relu(X_e2 @ W1_e2 + b1_e2) @ W2_e2 + b2_e2 # recompute trained predictions.
print("final loss:", round(losses_e2[-1], 4)); assert losses_e2[-1] < losses_e2[0]
plt.figure(figsize=(4.5, 3)); plt.scatter(X_e2.ravel(), y_e2.ravel(), label="target", color="black"); plt.plot(X_e2.ravel(), yhat_e2.ravel(), marker="o", label="network", color="teal")
plt.title("Easy 2: tiny ReLU network fit"); plt.legend(); plt.show()

▶ What you'll see: the piecewise-linear ReLU network bends toward the curved target.

👀 Takeaway: hidden-layer learning multiplies output gradients through weights and ReLU gates.

### Easy 3 — Train a softmax classifier from scratch

**Goal.** train a softmax classifier from scratch, because this is the next step after the scalar chain-rule warm-up. We build it in one compact runnable cell.

In [ ]:
X_e3 = np.array([[2.0, 1.0], [1.5, 1.2], [-1.0, -1.5], [-1.3, -0.8]]) # create separable points.
y_idx_e3 = np.array([0, 0, 1, 1]) # assign class labels.
Y_e3 = np.eye(2)[y_idx_e3] # one-hot targets.
W_e3 = np.zeros((2, 2)); b_e3 = np.zeros(2) # initialize classifier.
losses_e3 = [] # store cross-entropy losses.
for step_e3 in range(200): # train with full-batch gradient descent.
    logits_e3 = X_e3 @ W_e3 + b_e3; probs_e3 = softmax(logits_e3) # forward pass.
    loss_e3 = -np.mean(np.sum(Y_e3 * np.log(probs_e3 + 1e-12), axis=1)) # mean cross-entropy.
    dlogits_e3 = (probs_e3 - Y_e3) / len(X_e3) # softmax-cross-entropy gradient.
    W_e3 -= 0.3 * (X_e3.T @ dlogits_e3); b_e3 -= 0.3 * np.sum(dlogits_e3, axis=0) # update parameters.
    losses_e3.append(float(loss_e3)) # record loss.
probs_e3 = softmax(X_e3 @ W_e3 + b_e3); preds_e3 = np.argmax(probs_e3, axis=1) # final predictions.
print("probabilities:\n", np.round(probs_e3, 3)); assert np.array_equal(preds_e3, y_idx_e3)
plt.figure(figsize=(4.5, 3)); plt.plot(losses_e3, color="navy")
plt.title("Easy 3: softmax training loss"); plt.xlabel("step"); plt.ylabel("cross-entropy"); plt.show()

▶ What you'll see: cross-entropy falls quickly because the data are linearly separable.

👀 Takeaway: softmax backprop turns class probability errors directly into logit gradients.

### Easy 4 — Check a gradient with finite differences

**Goal.** check a gradient with finite differences, because this is the next step after the scalar chain-rule warm-up. We build it in one compact runnable cell.

In [ ]:
x_e4 = np.array([1.5, -0.5]); w_e4 = np.array([1.8, -0.7]); b_e4 = 0.7; v_e4 = 0.8; y_e4 = 2.0 # define the scalar network.
z_e4 = float(w_e4 @ x_e4 + b_e4); h_e4 = max(0.0, z_e4); yhat_e4 = v_e4 * h_e4 # forward pass.
analytic_w0_e4 = (yhat_e4 - y_e4) * v_e4 * (1.0 if z_e4 > 0 else 0.0) * x_e4[0] # analytic chain-rule gradient.
eps_e4 = 1e-5 # tiny perturbation.
w_plus_e4 = w_e4.copy(); w_plus_e4[0] += eps_e4 # plus perturbation.
w_minus_e4 = w_e4.copy(); w_minus_e4[0] -= eps_e4 # minus perturbation.
loss_plus_e4 = 0.5 * (v_e4 * max(0.0, float(w_plus_e4 @ x_e4 + b_e4)) - y_e4) ** 2 # plus loss.
loss_minus_e4 = 0.5 * (v_e4 * max(0.0, float(w_minus_e4 @ x_e4 + b_e4)) - y_e4) ** 2 # minus loss.
numeric_w0_e4 = (loss_plus_e4 - loss_minus_e4) / (2 * eps_e4) # centered finite difference.
print("analytic/numeric:", round(analytic_w0_e4, 6), round(numeric_w0_e4, 6)); assert abs(analytic_w0_e4 - numeric_w0_e4) < 1e-8
plt.figure(figsize=(4, 3)); plt.bar(["analytic", "numeric"], [analytic_w0_e4, numeric_w0_e4], color=["teal", "orange"])
plt.title("Easy 4: gradient check"); plt.ylabel("dL/dw0"); plt.show()

▶ What you'll see: the analytic and numeric bars are visually identical.

👀 Takeaway: finite differences are slow but useful for checking scratch backprop formulas.

### Easy 5 — Visualize gradient flow through depth

**Goal.** visualize gradient flow through depth, because this is the next step after the scalar chain-rule warm-up. We build it in one compact runnable cell.

In [ ]:
layers_e5 = np.arange(1, 13) # define depths from 1 to 12 layers.
small_deriv_e5 = 0.7; large_deriv_e5 = 1.2 # compare local derivatives below and above one.
vanish_e5 = small_deriv_e5 ** layers_e5 # repeated shrinkage.
explode_e5 = large_deriv_e5 ** layers_e5 # repeated amplification.
print("depth 12 shrink/grow:", round(float(vanish_e5[-1]), 4), round(float(explode_e5[-1]), 4))
assert round(float(vanish_e5[-1]), 4) == 0.0138
plt.figure(figsize=(4.5, 3)); plt.plot(layers_e5, vanish_e5, marker="o", label="0.7^depth"); plt.plot(layers_e5, explode_e5, marker="s", label="1.2^depth")
plt.title("Easy 5: repeated local derivatives"); plt.xlabel("depth"); plt.ylabel("gradient multiplier"); plt.legend(); plt.show()

▶ What you'll see: multiplying local derivatives repeatedly can make gradients tiny or large.

👀 Takeaway: deep learning stability is partly about keeping products of local derivatives in a useful range.

## 🔴 Advanced

### Advanced 1 — Train XOR with a hidden layer

**Goal.** train xor with a hidden layer, because advanced backprop work is mostly about stability, capacity, stochasticity, and hardware cost. We build it in one compact runnable cell.

In [ ]:
X_a1 = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]]) # create XOR inputs.
y_a1 = np.array([[0.], [1.], [1.], [0.]]) # create XOR targets.
rng_a1 = np.random.default_rng(11) # reproducible initialization.
W1_a1 = rng_a1.normal(scale=0.8, size=(2, 4)); b1_a1 = np.zeros(4) # hidden layer.
W2_a1 = rng_a1.normal(scale=0.8, size=(4, 1)); b2_a1 = np.zeros(1) # output layer.
losses_a1 = [] # store training losses.
for step_a1 in range(5000): # train the hidden-layer network.
    Z1_a1 = X_a1 @ W1_a1 + b1_a1; H1_a1 = np.tanh(Z1_a1); logits_a1 = H1_a1 @ W2_a1 + b2_a1 # forward pass.
    pred_a1 = 1 / (1 + np.exp(-logits_a1)) # sigmoid probability.
    loss_a1 = -np.mean(y_a1 * np.log(pred_a1 + 1e-12) + (1 - y_a1) * np.log(1 - pred_a1 + 1e-12)) # binary cross-entropy.
    dlogits_a1 = (pred_a1 - y_a1) / len(X_a1) # sigmoid-cross-entropy gradient.
    dW2_a1 = H1_a1.T @ dlogits_a1; db2_a1 = np.sum(dlogits_a1, axis=0) # output gradients.
    dZ1_a1 = (dlogits_a1 @ W2_a1.T) * (1 - H1_a1 ** 2) # tanh hidden gradient.
    dW1_a1 = X_a1.T @ dZ1_a1; db1_a1 = np.sum(dZ1_a1, axis=0) # hidden gradients.
    W2_a1 -= 0.8 * dW2_a1; b2_a1 -= 0.8 * db2_a1; W1_a1 -= 0.8 * dW1_a1; b1_a1 -= 0.8 * db1_a1 # update.
    if step_a1 % 50 == 0: losses_a1.append(float(loss_a1)) # record loss.
pred_a1 = 1 / (1 + np.exp(-(np.tanh(X_a1 @ W1_a1 + b1_a1) @ W2_a1 + b2_a1))) # final probabilities.
labels_a1 = (pred_a1 > 0.5).astype(int) # threshold probabilities.
print("probabilities:", np.round(pred_a1.ravel(), 3)); assert np.array_equal(labels_a1, y_a1.astype(int))
plt.figure(figsize=(4.5, 3)); plt.plot(np.arange(len(losses_a1)) * 50, losses_a1, color="purple")
plt.title("Advanced 1: XOR training loss"); plt.xlabel("step"); plt.ylabel("binary cross-entropy"); plt.show()

▶ What you'll see: the nonlinear hidden layer reduces XOR loss to a small value.

👀 Takeaway: backprop can assign credit through hidden nonlinear features that make XOR learnable.

### Advanced 2 — Compare initialization scales

**Goal.** compare initialization scales, because advanced backprop work is mostly about stability, capacity, stochasticity, and hardware cost. We build it in one compact runnable cell.

In [ ]:
X_a2 = np.linspace(-1, 1, 20).reshape(-1, 1) # create one-dimensional inputs.
y_a2 = np.sin(3 * X_a2) # create a smooth target.
scales_a2 = [0.05, 1.5] # compare modest and too-large initialization scales.
curves_a2 = [] # store loss curves.
for scale_a2 in scales_a2: # train one model per scale.
    rng_a2 = np.random.default_rng(22); W1_a2 = rng_a2.normal(scale=scale_a2, size=(1, 12)); b1_a2 = np.zeros(12); W2_a2 = rng_a2.normal(scale=scale_a2, size=(12, 1)); b2_a2 = np.zeros(1)
    losses_scale_a2 = [] # store this run's losses.
    for step_a2 in range(500): # train two-layer ReLU network.
        Z1_a2 = X_a2 @ W1_a2 + b1_a2; H1_a2 = relu(Z1_a2); yhat_a2 = H1_a2 @ W2_a2 + b2_a2 # forward pass.
        dY_a2 = (yhat_a2 - y_a2) / len(X_a2); dW2_a2 = H1_a2.T @ dY_a2; db2_a2 = np.sum(dY_a2, axis=0) # output grads.
        dZ1_a2 = (dY_a2 @ W2_a2.T) * relu_grad(Z1_a2); dW1_a2 = X_a2.T @ dZ1_a2; db1_a2 = np.sum(dZ1_a2, axis=0) # hidden grads.
        W2_a2 -= 0.05 * dW2_a2; b2_a2 -= 0.05 * db2_a2; W1_a2 -= 0.05 * dW1_a2; b1_a2 -= 0.05 * db1_a2 # update.
        if step_a2 % 10 == 0: losses_scale_a2.append(float(mse_loss(yhat_a2, y_a2))) # record loss.
    curves_a2.append(losses_scale_a2) # save curve.
print("final losses:", [round(c[-1], 4) for c in curves_a2]); assert curves_a2[1][0] > curves_a2[0][0]
plt.figure(figsize=(4.5, 3)); plt.semilogy(curves_a2[0], label="scale=0.05"); plt.semilogy(curves_a2[1], label="scale=1.5")
plt.title("Advanced 2: initialization scale"); plt.xlabel("checkpoint"); plt.ylabel("half-MSE (log)"); plt.legend(); plt.show()

▶ What you'll see: the large-scale run starts with much larger errors and different optimization behavior.

👀 Takeaway: initialization scale changes the numerical path that gradients must travel.

### Advanced 3 — Add L2 regularization to backprop

**Goal.** add l2 regularization to backprop, because advanced backprop work is mostly about stability, capacity, stochasticity, and hardware cost. We build it in one compact runnable cell.

In [ ]:
X_a3 = np.array([[-2.], [-1.], [0.], [1.], [2.]]) # create regression inputs.
y_a3 = np.array([[-3.8], [-1.9], [0.1], [2.1], [3.9]]) # create nearly linear targets.
lams_a3 = [0.0, 0.2] # compare no regularization with L2 regularization.
weights_a3 = [] # store final weights.
for lam_a3 in lams_a3: # train one model per lambda.
    w_a3 = np.array([[0.0]]); b_a3 = np.array([0.0]) # initialize parameters.
    for step_a3 in range(150): # train full-batch gradient descent.
        pred_a3 = X_a3 @ w_a3 + b_a3; d_pred_a3 = (pred_a3 - y_a3) / len(X_a3) # forward and loss derivative.
        dw_a3 = X_a3.T @ d_pred_a3 + lam_a3 * w_a3; db_a3 = np.sum(d_pred_a3, axis=0) # add L2 derivative to weight gradient.
        w_a3 -= 0.08 * dw_a3; b_a3 -= 0.08 * db_a3 # update parameters.
    weights_a3.append(float(w_a3[0, 0])) # store final slope.
print("final slopes:", np.round(weights_a3, 3)); assert abs(weights_a3[1]) < abs(weights_a3[0])
plt.figure(figsize=(4.5, 3)); plt.bar(["λ=0", "λ=0.2"], weights_a3, color=["gray", "teal"])
plt.title("Advanced 3: L2 shrinks weights"); plt.ylabel("learned slope"); plt.show()

▶ What you'll see: the regularized model learns a smaller-magnitude slope.

👀 Takeaway: regularization enters backprop as an extra local gradient term that constrains capacity.

### Advanced 4 — Minibatch gradients estimate full gradients

**Goal.** minibatch gradients estimate full gradients, because advanced backprop work is mostly about stability, capacity, stochasticity, and hardware cost. We build it in one compact runnable cell.

In [ ]:
rng_a4 = np.random.default_rng(44) # reproducible synthetic data.
X_a4 = rng_a4.normal(size=(40, 2)) # create 40 examples with two features.
true_w_a4 = np.array([[1.5], [-2.0]]) # true linear weights.
y_a4 = X_a4 @ true_w_a4 + 0.3 # targets without noise.
w_a4 = np.array([[0.2], [0.1]]); b_a4 = np.array([0.0]) # current model parameters.
pred_full_a4 = X_a4 @ w_a4 + b_a4; dw_full_a4 = X_a4.T @ ((pred_full_a4 - y_a4) / len(X_a4)) # full gradient.
batch_idx_a4 = np.array([0, 3, 7, 11, 19]) # deterministic minibatch.
X_batch_a4 = X_a4[batch_idx_a4]; y_batch_a4 = y_a4[batch_idx_a4] # slice minibatch.
pred_batch_a4 = X_batch_a4 @ w_a4 + b_a4; dw_batch_a4 = X_batch_a4.T @ ((pred_batch_a4 - y_batch_a4) / len(X_batch_a4)) # minibatch gradient.
cos_grad_a4 = float(np.dot(dw_full_a4.ravel(), dw_batch_a4.ravel()) / (np.linalg.norm(dw_full_a4) * np.linalg.norm(dw_batch_a4))) # directional agreement.
print("full:", np.round(dw_full_a4.ravel(), 3), "batch:", np.round(dw_batch_a4.ravel(), 3), "cos:", round(cos_grad_a4, 3)); assert cos_grad_a4 > 0
plt.figure(figsize=(4.5, 3)); xpos_a4 = np.arange(2); width_a4 = 0.35
plt.bar(xpos_a4 - width_a4/2, dw_full_a4.ravel(), width_a4, label="full", color="teal"); plt.bar(xpos_a4 + width_a4/2, dw_batch_a4.ravel(), width_a4, label="minibatch", color="orange")
plt.axhline(0, color="black", linewidth=0.8); plt.xticks(xpos_a4, ["w0", "w1"]); plt.title("Advanced 4: minibatch gradient estimate"); plt.legend(); plt.show()

▶ What you'll see: the minibatch gradient differs from the full gradient but points broadly downhill.

👀 Takeaway: stochastic backprop trades exact full-data gradients for cheaper noisy estimates.

### Advanced 5 — Track activation memory across a model

**Goal.** track activation memory across a model, because advanced backprop work is mostly about stability, capacity, stochasticity, and hardware cost. We build it in one compact runnable cell.

In [ ]:
batch_sizes_a5 = np.array([16, 32, 64, 128]) # define batch sizes to compare.
widths_a5 = np.array([64, 128, 128, 32]) # define hidden widths for four layers.
bytes_per_float_a5 = 4 # assume float32 activations.
mem_by_batch_a5 = [] # store total memory per batch size.
for batch_a5 in batch_sizes_a5: # loop over batches.
    mem_kb_a5 = float(np.sum(batch_a5 * widths_a5 * bytes_per_float_a5) / 1024) # cached activation memory.
    mem_by_batch_a5.append(mem_kb_a5) # store memory.
per_layer_a5 = 32 * widths_a5 * bytes_per_float_a5 / 1024 # per-layer memory for batch 32.
print("memory KB:", np.round(mem_by_batch_a5, 1)); assert mem_by_batch_a5[1] == 44.0
plt.figure(figsize=(4.5, 3)); plt.plot(batch_sizes_a5, mem_by_batch_a5, marker="o", color="purple")
plt.title("Advanced 5: activation memory by batch"); plt.xlabel("batch size"); plt.ylabel("cached activation KB"); plt.show()

▶ What you'll see: cached activation memory grows linearly with batch size.

👀 Takeaway: backprop's exact gradients require cached activations, so depth, width, and batch size become practical limits.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Backprop sends credit backward by multiplying local derivatives along the computational path.

A forward pass caches local quantities. The backward pass multiplies those cached derivatives to move the weights that caused the loss. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits
from sklearn.datasets import make_blobs
from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(6)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def split_scale(X, y):
    if len(y) < 20:
        scaler = StandardScaler()
        x_scaled = scaler.fit_transform(X)
        return x_scaled, x_scaled, y.copy(), y.copy(), scaler

    x_train, x_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=y,
    )
    rng = np.random.default_rng(606)
    if len(y_train) > 600:
        train_idx = rng.choice(len(y_train), size=600, replace=False)
        y_train = y_train[train_idx]
        x_train = x_train[train_idx]
    if len(y_test) > 300:
        test_idx = rng.choice(len(y_test), size=300, replace=False)
        y_test = y_test[test_idx]
        x_test = x_test[test_idx]

    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train)
    x_test = scaler.transform(x_test)
    return x_train, x_test, y_train, y_test, scaler


def one_hot(y, classes):
    out = np.zeros((len(y), classes))
    out[np.arange(len(y)), y.astype(int)] = 1.0
    return out


def stable_softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=1, keepdims=True)


def activation_forward(z, name):
    if name == "relu":
        return np.maximum(0.0, z)
    if name == "tanh":
        return np.tanh(z)
    if name == "sigmoid":
        return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        return 0.5 * z * (1.0 + np.tanh(c * (z + 0.044715 * z**3)))
    return z


def activation_backward(z, name):
    if name == "relu":
        return (z > 0.0).astype(float)
    if name == "tanh":
        h = np.tanh(z)
        return 1.0 - h**2
    if name == "sigmoid":
        h = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        return h * (1.0 - h)
    if name == "gelu":
        c = math.sqrt(2.0 / math.pi)
        u = c * (z + 0.044715 * z**3)
        t = np.tanh(u)
        sech2 = 1.0 - t**2
        return 0.5 * (1.0 + t) + 0.5 * z * sech2 * c * (1.0 + 3.0 * 0.044715 * z**2)
    return np.ones_like(z)


def initialize_mlp(n_features, n_hidden, n_classes, seed):
    rng = np.random.default_rng(seed)
    w1 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_features)), size=(n_features, n_hidden))
    b1 = np.zeros(n_hidden)
    w2 = rng.normal(0.0, math.sqrt(2.0 / max(1, n_hidden)), size=(n_hidden, n_classes))
    b2 = np.zeros(n_classes)
    return {"w1": w1, "b1": b1, "w2": w2, "b2": b2}


def forward(params, X, activation):
    z1 = X @ params["w1"] + params["b1"]
    h1 = activation_forward(z1, activation)
    logits = h1 @ params["w2"] + params["b2"]
    probs = stable_softmax(logits)
    cache = {"z1": z1, "h1": h1, "logits": logits, "probs": probs}
    return probs, cache


def compute_loss(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    eps = 1e-9

    if loss_name == "mse":
        return float(np.mean((probs - targets) ** 2))

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        margins = np.maximum(0.0, probs - correct[:, None] + 0.2)
        margins[np.arange(len(y)), y] = 0.0
        return float(np.mean(np.sum(margins, axis=1)))

    return float(-np.mean(np.log(probs[np.arange(len(y)), y] + eps)))


def output_gradient(probs, y, loss_name):
    classes = probs.shape[1]
    targets = one_hot(y, classes)
    n = max(1, len(y))

    if loss_name == "mse":
        return 2.0 * (probs - targets) / (n * classes)

    if loss_name == "hinge":
        correct = probs[np.arange(len(y)), y]
        active = probs - correct[:, None] + 0.2 > 0.0
        active[np.arange(len(y)), y] = False
        grad = active.astype(float)
        grad[np.arange(len(y)), y] = -grad.sum(axis=1)
        return grad / n

    return (probs - targets) / n


def train_tiny_mlp(
    X,
    y,
    hidden=24,
    epochs=25,
    lr=0.12,
    activation="relu",
    loss_name="ce",
    seed=0,
):
    x_train, x_test, y_train, y_test, scaler = split_scale(X, y)
    classes = int(np.max(y)) + 1
    params = initialize_mlp(x_train.shape[1], hidden, classes, seed)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        probs, cache = forward(params, x_train, activation)
        grad_logits = output_gradient(probs, y_train, loss_name)
        grad_w2 = cache["h1"].T @ grad_logits
        grad_b2 = grad_logits.sum(axis=0)
        grad_h = grad_logits @ params["w2"].T
        grad_z1 = grad_h * activation_backward(cache["z1"], activation)
        grad_w1 = x_train.T @ grad_z1
        grad_b1 = grad_z1.sum(axis=0)

        params["w1"] = params["w1"] - lr * grad_w1
        params["b1"] = params["b1"] - lr * grad_b1
        params["w2"] = params["w2"] - lr * grad_w2
        params["b2"] = params["b2"] - lr * grad_b2

        train_probs, _ = forward(params, x_train, activation)
        val_probs, _ = forward(params, x_test, activation)
        train_pred = train_probs.argmax(axis=1)
        val_pred = val_probs.argmax(axis=1)
        history["train_loss"].append(compute_loss(train_probs, y_train, loss_name))
        history["val_loss"].append(compute_loss(val_probs, y_test, loss_name))
        history["train_acc"].append(float(accuracy_score(y_train, train_pred)))
        history["val_acc"].append(float(accuracy_score(y_test, val_pred)))

    result = {
        "params": params,
        "history": history,
        "scaler": scaler,
        "x_test": x_test,
        "y_test": y_test,
        "activation": activation,
        "loss_name": loss_name,
    }
    return result


def predict_tiny(model, X_raw):
    X = model["scaler"].transform(X_raw)
    probs, _ = forward(model["params"], X, model["activation"])
    return probs.argmax(axis=1)


def evaluate_ladder(hidden=24, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=0):
    rows = []
    models = []
    for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
        model = train_tiny_mlp(
            X,
            y,
            hidden=hidden,
            epochs=epochs,
            lr=lr,
            activation=activation,
            loss_name=loss_name,
            seed=seed + idx,
        )
        metric = model["history"]["val_acc"][-1]
        loss_value = model["history"]["val_loss"][-1]
        rows.append({"rung": idx, "name": name, "accuracy": metric, "loss": loss_value})
        models.append(model)
    return rows, models


def print_rows(rows, metric="accuracy"):
    print("rung | dataset | accuracy | loss")
    for row in rows:
        print(f"D{row['rung']} | {row['name']} | {row['accuracy']:.3f} | {row['loss']:.3f}")


def plot_decision_panel(ax, model, X, y, title):
    if X.shape[1] != 2:
        side = int(math.sqrt(X.shape[1]))
        if side * side == X.shape[1]:
            ax.imshow(X[0].reshape(side, side), cmap="gray")
            pred = predict_tiny(model, X[:1])[0]
            ax.set_title(f"{title}\ny={int(y[0])}, pred={int(pred)}")
        else:
            ax.plot(X[0])
            ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        return

    x_min = X[:, 0].min() - 0.8
    x_max = X[:, 0].max() + 0.8
    y_min = X[:, 1].min() - 0.8
    y_max = X[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 40), np.linspace(y_min, y_max, 40))
    grid = np.c_[xx.ravel(), yy.ravel()]
    zz = predict_tiny(model, grid).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, levels=np.arange(int(np.max(y)) + 2) - 0.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, s=16, edgecolor="k", linewidth=0.2)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])


def closing_figure(rows, models, metric="accuracy"):
    rungs = clf_digits_ladder()
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, model, rung in zip(axes[0], models, rungs):
        name, X, y = rung
        plot_decision_panel(ax, model, X, y, name.split(" (")[0])

    xs = [row["rung"] for row in rows]
    ys = [row[metric] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("ladder rung")
    axes[1, 0].set_ylabel(metric)
    axes[1, 0].set_title(f"{metric} vs ladder difficulty")
    for extra in axes[1, 1:]:
        extra.axis("off")
    fig.tight_layout()
    plt.show()


def plot_history(models, metric="val_acc"):
    plt.figure(figsize=(7, 4))
    for idx, model in enumerate(models, start=1):
        plt.plot(model["history"][metric], label=f"D{idx}")
    plt.xlabel("epoch")
    plt.ylabel(metric)
    plt.legend(ncol=3)
    plt.title(f"Training trace: {metric}")
    plt.show()


## Build the concept once on D1

The lesson formula is

$$\frac{\partial L}{\partial W_1}=\frac{\partial L}{\partial h}\frac{\partial h}{\partial z_1}\frac{\partial z_1}{\partial W_1}$$

We plug in the lesson's own numbers and assert the exact rounded values before using the method on the ladder.

In [ ]:

def manual_backprop_step():
    x = np.array([1.5, -0.5])
    w = np.array([1.8, -0.7])
    b = 0.7
    z = float(w @ x + b)
    h = max(0.0, z)
    dloss_dh = 0.5
    dh_dz = 1.0 if z > 0.0 else 0.0
    dz_dw = x
    grad_w = dloss_dh * dh_dz * dz_dw
    return z, h, grad_w

z, h, grad_w = manual_backprop_step()
updated = 2.0 - 0.090 * 1.800
prob = math.exp(z) / (math.exp(z) + math.exp(0.4))
normalized = (z - 1.0) / math.sqrt(0.250 + 0.00001)
memory_kb = 2 * 128 * 4 / 1024

assert round(z, 3) == 3.750
assert round(h, 3) == 3.750
assert np.allclose(grad_w, np.array([0.75, -0.25]))
assert round(updated, 3) == 1.838
assert round(math.exp(z), 3) == 42.521
assert round(prob, 3) == 0.966
assert round(normalized, 3) == 5.500
assert round(memory_kb, 3) == 1.000

print("chain-rule dL/dW:", grad_w)


## Package the reusable method

The same tiny MLP code above performs a real forward pass, computes a real loss, backpropagates analytic gradients, and updates weights on CPU.

In [ ]:
rungs = clf_digits_ladder()
print("Reusable method: train_tiny_mlp + forward + analytic backward pass")
print("Rungs ready:", len(rungs))

## The dataset ladder

D1 is XOR, then blobs, noisy moons, real sklearn digits, and D5 digits with feature plus label noise. The method sees one feature matrix and one class vector at every rung.

In [ ]:
for idx, (name, X, y) in enumerate(clf_digits_ladder(), start=1):
    labels, counts = np.unique(y, return_counts=True)
    print(f"D{idx}: {name}")
    print("  X shape:", X.shape)
    print("  classes:", labels.tolist())
    print("  first counts:", counts[:5].tolist())
    print("  first row sample:", np.round(X[0, : min(8, X.shape[1])], 3))

## Run the same method across D1–D5

The tracked metric is loss.

In [ ]:
rows, models = evaluate_ladder(hidden=28, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=64)
print_rows(rows, metric="loss")

## Results visualization

The closing figure has small-multiple output artifacts for every rung plus the metric curve from D1 to D5. The second plot shows train/validation history so local steps can be compared with generalization.

In [ ]:
closing_figure(rows, models, metric="loss")
plot_history(models, metric="val_loss")

## Pitfall on the hardest rung

D5 is real digits with added feature and label noise, so the pitfall has to show up in a genuinely harder training run rather than a toy picture.

In [ ]:

name, X, y = clf_digits_ladder()[-1]
one_step = train_tiny_mlp(X, y, hidden=32, epochs=1, lr=0.12, activation="relu", loss_name="ce", seed=645)
validated = train_tiny_mlp(X, y, hidden=32, epochs=25, lr=0.12, activation="relu", loss_name="ce", seed=645)
print("one correct step D5 val loss:", round(one_step["history"]["val_loss"][-1], 3))
print("validation-tracked D5 val loss:", round(validated["history"]["val_loss"][-1], 3))
plot_history([one_step, validated], metric="val_loss")


## Evaluate it + Practice

- Metric: inspect D5 loss and compare with a no-skill baseline near the majority-class rate.
- Sanity check: D1 XOR should be learnable by a hidden-layer MLP but not by one linear gate.
- Ablation: reduce width, saturate activations, or use the wrong loss and watch the metric degrade.
- Failure signals: diverging loss, flat validation curves, unstable softmax sums, or gradients with unexpected shapes.

Practice prompts:
1. Change hidden width and replot the D1–D5 curve.

2. Replace ReLU with tanh or GELU and compare D5.

3. Lower the D5 label-noise rate in `clf_digits_ladder()` and predict how the curve changes.